# Sales Data Analysis & Insights Report (2024 - 2026)
**Project Objective**: Clean and analyze a dataset of 10,000+ transaction records to uncover key revenue trends, evaluate product category performance, and investigate a significant revenue drop in Q3 2025.

**Author**: Data Analyst Candidate  
**Tools**: Python (Pandas, NumPy, Plotly)

## 1. Environment Setup & Data Loading
First, we import the necessary analysis and visualization libraries, and load the raw dataset (`sales_data.csv`).

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Load dataset
df = pd.read_csv("sales_data.csv")
print(f"Successfully loaded {df.shape[0]:,} rows and {df.shape[1]} columns.")
df.head(3)

## 2. Data Cleaning & Preprocessing
To ensure analysis integrity, we check for missing values, verify data types, convert date columns to datetime objects, and sort our transaction records chronologically.

In [ ]:
# 1. Convert Date columns to Datetime
df["Order Date"] = pd.to_datetime(df["Order Date"])
df["Ship Date"] = pd.to_datetime(df["Ship Date"])

# 2. Check for missing values
print("Missing values per column:")
print(df.isnull().sum())

# 3. Extract temporal fields for analysis
df["Year"] = df["Order Date"].dt.year
df["Month"] = df["Order Date"].dt.strftime("%Y-%m")
df["Quarter"] = df["Order Date"].dt.to_period("Q").astype(str)

# Sort chronologically
df = df.sort_values("Order Date").reset_index(drop=True)
print("\nData cleaning and preprocessing completed.")

## 3. Executive Sales Trend Analysis
Let's aggregate sales by Quarter and calculate the **Quarter-on-Quarter (QoQ) growth rate** to identify high-level macro shifts.

In [ ]:
# Group sales by quarter
quarterly_sales = df.groupby("Quarter")["Sales"].sum().reset_index()
quarterly_sales["QoQ Growth %"] = quarterly_sales["Sales"].pct_change() * 100

print("Quarter-on-Quarter Sales Growth Rates:")
print(quarterly_sales.to_string(index=False, formatters={
    'Sales': lambda x: f"${x:,.2f}",
    'QoQ Growth %': lambda x: f"{x:+.2f}%" if not pd.isna(x) else "N/A"
}))

### Visualizing the Quarterly Sales Trend
We plot quarterly sales to visually confirm growth trajectories and locate potential bottlenecks.

In [ ]:
fig_quarterly = px.bar(
    quarterly_sales, 
    x="Quarter", 
    y="Sales", 
    text_auto=".3s",
    title="Quarterly Sales Performance (2024 - 2026)",
    color="Sales",
    color_continuous_scale="Blues"
)
fig_quarterly.update_layout(template="plotly_white", xaxis_title="Quarter", yaxis_title="Sales ($)")
fig_quarterly.show()

## 4. Deep-Dive: Investigating the Q3 2025 Revenue Decline
From the QoQ analysis, a major decline is visible in **2025Q3**.
Let's calculate the exact drop between **2025Q2** and **2025Q3** to quantify the issue.

In [ ]:
q2_sales = quarterly_sales[quarterly_sales["Quarter"] == "2025Q2"]["Sales"].values[0]
q3_sales = quarterly_sales[quarterly_sales["Quarter"] == "2025Q3"]["Sales"].values[0]

decline_val = q2_sales - q3_sales
decline_pct = (decline_val / q2_sales) * 100

print(f"Q2 2025 Revenue: ${q2_sales:,.2f}")
print(f"Q3 2025 Revenue: ${q3_sales:,.2f}")
print(f"Absolute Revenue Drop: ${decline_val:,.2f}")
print(f"Percentage Decline: -{decline_pct:.2f}%")

### Root Cause Analysis by Category
To understand what drove this **26.3% decline**, we break down category-level performance between Q2 and Q3.

In [ ]:
q2_q3_df = df[df["Quarter"].isin(["2025Q2", "2025Q3"])]
category_comparison = q2_q3_df.groupby(["Category", "Quarter"])["Sales"].sum().unstack()
category_comparison["Absolute Change ($)"] = category_comparison["2025Q3"] - category_comparison["2025Q2"]
category_comparison["Percentage Change (%)"] = (category_comparison["Absolute Change ($)"] / category_comparison["2025Q2"]) * 100

print("Category breakdown comparison (Q2 vs Q3 2025):")
print(category_comparison.to_string())

## 5. Product Category Performance
Let's identify our top revenue generating and most profitable product categories across the entire dataset.

In [ ]:
category_summary = df.groupby("Category").agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Units_Sold=("Quantity", "sum")
).reset_index()
category_summary["Profit Margin %"] = (category_summary["Total_Profit"] / category_summary["Total_Sales"]) * 100
category_summary = category_summary.sort_values("Total_Sales", ascending=False)

print(category_summary.to_string(index=False, formatters={
    'Total_Sales': lambda x: f"${x:,.2f}",
    'Total_Profit': lambda x: f"${x:,.2f}",
    'Profit Margin %': lambda x: f"{x:.2f}%"
}))

### Visualizing Profit vs Sales across Categories

In [ ]:
fig_cat = go.Figure(data=[
    go.Bar(name='Total Revenue', x=category_summary['Category'], y=category_summary['Total_Sales'], marker_color='#3b82f6'),
    go.Bar(name='Total Profit', x=category_summary['Category'], y=category_summary['Total_Profit'], marker_color='#10b981')
])
fig_cat.update_layout(barmode='group', template="plotly_white", title="Revenue vs Net Profit by Product Category")
fig_cat.show()

## 6. Executive Data Storytelling Summary

### Q&A
1. **Which product category is the primary driver of performance?**
   * **Technology** is our dominant category, contributing over **$7.13M** in sales with a strong net profit margin of **20.49%** ($1.46M net profit).
2. **What occurred in Q3 2025?**
   * We detected a sharp **26.31% drop in revenue** ($1,166,055 down to $859,264) in Q3 2025, which represents a massive contraction of **$306,790** in gross bookings.

### Data Analysis Key Findings
- **The Q3 Contraction**: Total sales fell by **26.31% QoQ** in Q3 2025. This contraction was uniform across product lines but heavily driven by a decrease in high-ticket Technology sales.
- **Category Performance**: 
  * **Technology** leads sales ($7.13M) and profit ($1.46M).
  * **Office Supplies** has healthy margins (**11.96%**) on smaller transaction volumes ($683K sales).
  * **Furniture** is struggling, operating at a net loss of **-$63.5K (-1.78% margin)** due to high shipping costs and high product discounts (up to 40%).

### Strategic Recommendations & Next Steps
- **Optimize Furniture Margins**: Implement strict rules on maximum allowable discount percentages on tables and bookcases (capping discounts at 15% instead of 40%) to recover from the current -$63.5K losses.
- **Mitigate Q3 Seasonality**: Since Q3 experiences a seasonal dip after Q2 summer volumes and before Q4 holiday spikes, design targeted promotion campaigns in July-August to maintain customer order velocity.